# Committor of Benzamidine Unbinding from Trypsin (infretis StapleTIS)

Fits the committor q of benzamidine unbinding from trypsin with NPNEq (`CommittorNE`, CuPy) on the paths of an **infretis StapleTIS** run (GROMACS), using the order parameter and the CVs of `OP_new26.py`, then shows how q depends on them.

**The run:** an infretis folder (`infretis.toml`, `infretis_data.txt`, an `archive.h5` from `inft store_txt -order` or `load/`). The order parameter λ is the distance between the centre of mass of the pocket Cα atoms and that of the ligand, in Å. A (bound): λ ≤ `LAMBDA_A`, B (unbound): λ ≥ `LAMBDA_B` (default: the first and last interface).

**CVs** (`OP_new26.py`, `COM_Distance.calculate`; `order.txt` holds the time and then these 14 columns):

| Column | Name | Meaning |
|---|---|---|
| 1 | `lambda` | protein–ligand COM distance (Å), the order parameter |
| 2 | `angle` | angle between the ligand vector (from Asp189 Cα) and the Asp189 → Gly191 Cα axis (deg) |
| 3–5 | `rel_x`, `rel_y`, `rel_z` | ligand COM in a frame fixed to the Cα of residues 189, 191 and 221 (Å) |
| 6 | `d_triad_core` | His57 ND1 – Asp102 OD1 (Å) |
| 7 | `d_triad_attack` | Ser195 OG – His57 NE2 (Å) |
| 8–10 | `dA_99`, `dA_175`, `dA_216` | minimum ligand distance to residues 99, 175, 216 (pathway A, Å) |
| 11–13 | `dB_57`, `dB_195`, `dB_60` | minimum ligand distance to residues 57, 195, 60 (pathway B, Å) |
| 14 | `n_hbonds` | H-bonds between the ligand and Asp189 |

All of them are even under time reversal (no velocities), so the mirrored pieces can use them unchanged.

**Training data** (`tis_data.prepare_data` → `infretis_data.py`; `docs/npneq_tis_notes.md` §3, 9, 15): each (path, ensemble) pair gives the piece from its first frame past the ensemble's M interface, and with `REVERSED` also the part before its last M crossing, read backwards. Each piece has the weight of its path in that ensemble (`'staple'` = path_f / path_w, `'rejections'` = path_f). The pieces of one path share its `group_traj` and count as one unit in the error estimates.

**No exact committor.** The checks are the convergence metrics (`zqs_z` ≲ 2) and the stopped observed-vs-predicted plot.


## 1. Imports

In [ ]:
import argparse
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cupy as cp

plt.rc('font', size=10)
plt.rc('axes', titlesize=11)
plt.rc('axes', labelsize=11)
plt.rc('legend', fontsize=9)

# Run from examples/: the CuPy optimalrcs package and the path readers of the examples
REPO = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.insert(0, f'{REPO}/optimalrcs-cupy/')
sys.path.insert(0, f'{REPO}/examples/')
import optimalrcs
import tis_data
import infretis_data
FIG = f'{REPO}/examples/figures_by_potential'


## 2. Settings

| Name | Meaning |
|---|---|
| `TIS_DIR`, `H5_FILE` | the infretis run folder and its order-parameter archive (`None`: read `load/<pnr>/order.txt`) |
| `LAMBDA_A`, `LAMBDA_B` | A: λ ≤ λ_A (bound), B: λ ≥ λ_B (unbound); `None`: first and last interface |
| `BURN_IN_FRACTION`, `PATHS_PER_ENSEMBLE` | skip the first part of `infretis_data.txt`; at most this many paths per ensemble, evenly spaced |
| `REVERSED` | also the mirrored pieces |
| `WEIGHTING` | `None` (the run's default: path_f / path_w with infinite swap), `'staple'` or `'rejections'` |
| `CV_NAMES` | the `order.txt` columns after the time, in the order `OP_new26.py` returns them |
| `CV_EXCLUDE` | columns not used as fit CVs (they are still plotted) |
| `DT_FRAME` | time between frames (ps): GROMACS timestep × subcycles; for the natural coordinate |
| `WARMUP_ITER`, `ITER_PER_CV`, `CHUNK` | warm-up iterations (wide envelope), main iterations per fit CV, iterations between plots |
| `HISTORY` | `None`, or lags d drawn per iteration for a history basis (y(t−d), y(t)) |


In [ ]:
TIS_DIR = f'{REPO}/examples/data/infretis_inputs/trypsin_benz'   # infretis.toml, infretis_data.txt, archive.h5 (or load/)
H5_FILE = f'{TIS_DIR}/archive.h5'   # None: read load/<pnr>/order.txt
LAMBDA_A, LAMBDA_B = None, None     # None: first and last interface
BURN_IN_FRACTION = 0.3
PATHS_PER_ENSEMBLE = 3000           # None: all paths after the burn-in
REVERSED = True
WEIGHTING = None                    # None: the run's default; or 'staple' / 'rejections'

CV_NAMES = ['lambda', 'angle', 'rel_x', 'rel_y', 'rel_z', 'd_triad_core', 'd_triad_attack',
            'dA_99', 'dA_175', 'dA_216', 'dB_57', 'dB_195', 'dB_60', 'n_hbonds']
CV_EXCLUDE = []                     # e.g. ['d_triad_core', 'd_triad_attack'] to fit without the triad
CV_LABELS = {'lambda': r'$\lambda$ (Å)', 'angle': 'angle (deg)', 'rel_x': 'x local (Å)', 'rel_y': 'y local (Å)',
             'rel_z': 'z local (Å)', 'd_triad_core': 'His57–Asp102 (Å)', 'd_triad_attack': 'Ser195–His57 (Å)',
             'dA_99': 'd res 99 (Å)', 'dA_175': 'd res 175 (Å)', 'dA_216': 'd res 216 (Å)',
             'dB_57': 'd res 57 (Å)', 'dB_195': 'd res 195 (Å)', 'dB_60': 'd res 60 (Å)', 'n_hbonds': 'H-bonds to Asp189'}
DT_FRAME = 0.002 * 100              # ps per frame (timestep x subcycles of infretis.toml)

SEED = 0
WARMUP_ITER, WARMUP_ENVELOPE = 2000, 1
ITER_PER_CV = 1000                  # main iterations per fit CV (each iteration draws one CV)
CHUNK = 2500
ENVELOPE, GAMMA = 0.01, 0.1
NY = 6
HISTORY = None                      # e.g. [0, 1, 2, 4, 8]: lags of a history basis (y(t-d), y(t))

OUT = f'{FIG}/trypsin_benz/infretis_staple/{os.path.basename(TIS_DIR.rstrip("/"))}'
os.makedirs(OUT, exist_ok=True)
print('figures and results in', OUT)


## 3. The Run

The sampling settings, and per ensemble the window (L, M, R), the paths with a nonzero occupancy and the share of the occupancy with `path_w` ≠ 1 (where `'staple'` and `'rejections'` differ).

In [ ]:
settings = infretis_data.read_settings(TIS_DIR)
table = infretis_data.read_infretis_data(os.path.join(TIS_DIR, 'infretis_data.txt'), len(settings['interfaces']))
default_w, why = infretis_data.default_weighting(settings)
print(f"mode {settings['mode']}, infinite swap {settings['infinite_swap']}, {len(table['pnr'])} finished paths")
print(f"interfaces {settings['interfaces']}")
print(f"default weighting: {default_w} ({why})")
f, w = table['path_f'], table['path_w']
rows = []
for e in range(len(settings['interfaces'])):
    on = f[:, e] > 0
    rows.append(dict(ensemble=f'{e:03d}', window=tuple(round(v, 3) for v in infretis_data.ensemble_window(settings, e)),
                     paths=int(on.sum()), occupancy=f[on, e].sum(),
                     share_w_not_1=(f[on, e] * ~np.isclose(w[on, e], 1)).sum() / max(f[on, e].sum(), 1e-12)))
display(pd.DataFrame(rows).set_index('ensemble').round(3))


## 4. From Paths to Frames

The pieces, their weights, the A/B masks and all `order.txt` columns per frame (`d.X_all`, λ first). The fit CVs are the columns of `CV_NAMES` that are not in `CV_EXCLUDE` and not constant. Piece ends outside A and B are turns of the path (stopped ends).

In [ ]:
parser = argparse.ArgumentParser()
tis_data.add_data_arguments(parser)
argv = [TIS_DIR, '--burn-in-fraction', str(BURN_IN_FRACTION)]
if H5_FILE is not None:
    argv += ['--h5', H5_FILE]
if LAMBDA_A is not None:
    argv += ['--lambda-a', str(LAMBDA_A)]
if LAMBDA_B is not None:
    argv += ['--lambda-b', str(LAMBDA_B)]
if PATHS_PER_ENSEMBLE is not None:
    argv += ['--paths-per-ensemble', str(PATHS_PER_ENSEMBLE)]
if REVERSED:
    argv += ['--reversed']
if WEIGHTING is not None:
    argv += ['--weights', WEIGHTING]
args = parser.parse_args(argv)
d = tis_data.prepare_data(args)
lambda_a, lambda_b = args.lambda_a, args.lambda_b

if d.X_all.shape[1] != len(CV_NAMES):
    raise SystemExit(f'order.txt has {d.X_all.shape[1]} columns after the time, CV_NAMES has {len(CV_NAMES)}: edit CV_NAMES')
free = ~(d.boundary0 | d.boundary1)
cv_names = [n for j, n in enumerate(CV_NAMES) if n not in CV_EXCLUDE and np.ptp(d.X_all[free, j]) > 0]
cv_idx = [CV_NAMES.index(n) for n in cv_names]
X = d.X_all[:, cv_idx]                     # the fit CVs
W = d.path_weights                         # weight of every frame (its path's weight in the piece's ensemble, mean 1)
lam = d.lam
label = lambda n: CV_LABELS.get(n, n)


def y_edges(y, nbins):
    # integer CVs (counts): one bin per value; otherwise nbins over the central 99 %
    if np.allclose(y, np.round(y)) and np.ptp(y) <= 60:
        return np.arange(np.min(y) - 0.5, np.max(y) + 1.5)
    lo, hi = np.quantile(y, [0.005, 0.995])
    return np.linspace(lo, hi + 1e-9, nbins + 1)


b_all = d.boundary0 | d.boundary1
piece_end = np.r_[np.flatnonzero(np.diff(d.i_traj)), len(d.i_traj) - 1]
stop_end = np.zeros(len(b_all), bool)
stop_end[piece_end] = ~b_all[piece_end]       # a piece that ends outside A and B ends at a turn
longest = int(np.bincount(d.i_traj - d.i_traj.min()).max())
TRAP_LEN = max(2 ** 16, longest + 2 ** 15 + 1)  # A/B trap: longer than any piece plus the longest Z_q lag
print(f'{len(lam)} frames, {d.i_traj.max() + 1} pieces, {len(np.unique(d.group_traj))} paths; '
      f'{stop_end.sum()} pieces end at a turn, {len(piece_end) - stop_end.sum()} in A or B; longest {longest} frames')
print(f'fit CVs ({len(cv_names)}): {", ".join(cv_names)}')


**Where the frames are.** Left: weighted frames along λ (high at the interfaces, where pieces start; this is the path-sampling density, not the equilibrium one). Right: piece lengths. Below: every CV against λ (path-weighted histogram, log colour), which shows which CVs change along the unbinding coordinate and which vary independently of it.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.8))
ax1.hist(lam, bins=120, weights=W, histtype='step')
for v in settings['interfaces']:
    ax1.axvline(v, color='0.6', lw=0.6)
ax1.set(xlabel=label('lambda'), ylabel='weighted frames', yscale='log')
lengths = np.bincount(d.i_traj - d.i_traj.min())
ax2.hist(lengths, bins=np.logspace(0, np.log10(lengths.max() + 1), 60))
ax2.set(xscale='log', yscale='log', xlabel='piece length (frames)', ylabel='pieces')
fig.tight_layout(); fig.savefig(f'{OUT}/data_overview.png', dpi=110); plt.show()

others = [n for n in CV_NAMES if n != 'lambda']
ncol = 4
nrow = int(np.ceil(len(others) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.2 * nrow), squeeze=False)
for ax, n in zip(axes.ravel(), others):
    y = d.X_all[free, CV_NAMES.index(n)]
    H, xe, ye = np.histogram2d(lam[free], y, bins=(np.linspace(lam[free].min(), lam[free].max(), 61), y_edges(y, 50)),
                               weights=W[free])
    ax.pcolormesh(xe, ye, np.log10(np.where(H > 0, H, np.nan)).T, cmap='viridis')
    ax.set(xlabel=label('lambda'), ylabel=label(n))
for ax in axes.ravel()[len(others):]:
    ax.axis('off')
fig.tight_layout(); fig.savefig(f'{OUT}/cvs_vs_lambda.png', dpi=110); plt.show()


## 5. Fit the Committor

The fit is written out here: `CommittorNE` with the A/B masks, pieces (`i_traj`), times, path weights and paths (`group_traj`); A/B ends as traps (`set_fixed_traj_length_trap`) and turn ends as stopped ends (`set_stopped_traj_end`), so no window of the stopped Z_q is dropped. Each iteration draws one fit CV (`comp_y`). A warm-up with a wide envelope, then the main fit in chunks of `CHUNK` iterations (`fit_transform` continues from the current q); after each chunk the metrics and the TIS validation plots are shown and saved.

**Convergence:** `zqs_z` (worst |stopped Z_q| / standard error over q and lags) about 2 or less, with `|dx|` small. In the `tis` plots: the stopped Z_q per lag stays inside ±2, and the stopped observed-vs-predicted curve is on the diagonal for both random halves of the paths.


In [ ]:
MAX_ITER = ITER_PER_CV * len(cv_names)
PRINT_STEP = min(500, CHUNK)
X_cv = cp.asarray(X)


def comp_y():
    # the CV of an iteration: one of the fit CVs, at random
    return X_cv[:, np.random.randint(X_cv.shape[1])]


fit_kw = dict(ny=NY, print_step=PRINT_STEP, lazy_basis=True, save_min_metric='max_z_zq_stopped',
              metrics_print=('iter', 'delta_r2', 'max_z_zq_stopped', 'max_sd_zq_stopped', 'delta_x', 'time_elapsed'))
if HISTORY is not None:
    fit_kw.update(history_delta_t=list(HISTORY), history_type='y(t-d),y(t)')


def show(q, tag):
    # library plots: metrics and TIS validation (stopped Z_q, stopped obs vs pred); saved to OUT/fit/
    os.makedirs(f'{OUT}/fit', exist_ok=True)
    keep, plt.show = plt.show, (lambda *a, **k: None)
    try:
        q.plots_metrics(['delta_r2', 'max_z_zq_stopped', 'delta_x'])
        plt.gcf().savefig(f'{OUT}/fit/{tag}_metrics.png', dpi=100)
        q.plots_tis()
        plt.gcf().savefig(f'{OUT}/fit/{tag}_tis.png', dpi=100)
    finally:
        plt.show = keep
    plt.show()


np.random.seed(SEED)
cp.random.seed(SEED)
q = optimalrcs.CommittorNE(
    boundary0=d.boundary0,          # A (bound): q = 0
    boundary1=d.boundary1,          # B (unbound): q = 1
    i_traj=d.i_traj,                # piece of every frame
    t_traj=d.t_traj,                # time of every frame within its path
    path_weights=d.path_weights,    # weight of every frame
    group_traj=d.group_traj,        # path of every frame (one unit in the error estimates)
)
q.set_fixed_traj_length_trap(b_all, TRAP_LEN)
q.set_stopped_traj_end(stop_end)

t0 = time.time()
optimalrcs.envelope_scale = WARMUP_ENVELOPE
q.fit_transform(comp_y, gamma=GAMMA, max_iter=WARMUP_ITER, **fit_kw)
show(q, f'it{WARMUP_ITER}')

optimalrcs.envelope_scale = ENVELOPE
n_done = 0
while n_done < MAX_ITER:
    n_iter = min(CHUNK, MAX_ITER - n_done)
    q.fit_transform(comp_y, gamma=GAMMA, max_iter=n_iter, **fit_kw)
    n_done += n_iter
    show(q, f'it{WARMUP_ITER + n_done}')
r = cp.asnumpy(q.r_traj)
zqs = [float(v) for v in q.metrics_history['max_z_zq_stopped']]
print(f'{WARMUP_ITER + MAX_ITER} iterations in {(time.time() - t0) / 60:.1f} min; zqs_z final {zqs[-1]:.2f}, min {min(zqs[1:]):.2f}')
np.save(f'{OUT}/q_traj.npy', r.astype(np.float32))


## 6. q and λ

**Left:** q of a random subset of frames against λ (colour: piece time, so the start of pieces at the interfaces shows), with the path-weighted mean and the 10–90 % range per λ bin. A wide spread at fixed λ means λ alone is a poor reaction coordinate there: frames at the same distance have different fates. **Right:** the path-weighted distribution of q per λ bin (log colour).

In [ ]:
rng = np.random.default_rng(0)
sub = rng.choice(np.flatnonzero(free), size=min(60000, free.sum()), replace=False)
edges_l = np.linspace(lambda_a, lambda_b, 61)
mid_l = (edges_l[1:] + edges_l[:-1]) / 2
bl = np.clip(np.digitize(lam, edges_l) - 1, 0, len(mid_l) - 1)


def wquant(x, wt, qs):
    o = np.argsort(x); c = np.cumsum(wt[o]); c /= c[-1]
    return np.interp(qs, c, x[o])


mean_q, lo_q, hi_q = (np.full(len(mid_l), np.nan) for _ in range(3))
for k in range(len(mid_l)):
    s = free & (bl == k)
    if s.sum() > 20:
        mean_q[k] = np.sum(W[s] * r[s]) / np.sum(W[s])
        lo_q[k], hi_q[k] = wquant(r[s], W[s], [0.1, 0.9])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
sc = ax1.scatter(lam[sub], r[sub], c=d.t_traj[sub], s=1, alpha=0.3, cmap='viridis', rasterized=True)
fig.colorbar(sc, ax=ax1, label='time in path (frames)')
ax1.plot(mid_l, mean_q, 'k-', lw=2, label='weighted mean')
ax1.fill_between(mid_l, lo_q, hi_q, color='k', alpha=0.15, label='10–90 %')
for v in settings['interfaces']:
    ax1.axvline(v, color='0.7', lw=0.5)
ax1.set(xlabel=label('lambda'), ylabel='$q$'); ax1.legend(loc='upper left')
H, xe, ye = np.histogram2d(lam[free], r[free], bins=(60, 50), range=((lambda_a, lambda_b), (0, 1)), weights=W[free])
H = H / np.maximum(H.sum(axis=1, keepdims=True), 1e-300)
im = ax2.pcolormesh(xe, ye, np.log10(np.where(H > 0, H, np.nan)).T, cmap='magma')
fig.colorbar(im, ax=ax2, label=r'$\log_{10} P(q \mid \lambda)$')
ax2.set(xlabel=label('lambda'), ylabel='$q$')
fig.tight_layout(); fig.savefig(f'{OUT}/q_vs_lambda.png', dpi=110); plt.show()


**q along the paths.** q against time for a few random pieces that end in B (red), in A (blue) and at a turn (grey). A good committor rises steadily on the way to B and falls on the way back to A, without jumps.

In [ ]:
starts = np.r_[0, piece_end[:-1] + 1]
end_in = np.where(d.boundary1[piece_end], 'B', np.where(d.boundary0[piece_end], 'A', 'turn'))
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, kind, col in zip(axes, ('B', 'A', 'turn'), ('tab:red', 'tab:blue', '0.4')):
    pick = np.flatnonzero(end_in == kind)
    for k in rng.choice(pick, size=min(15, len(pick)), replace=False) if len(pick) else []:
        sl = slice(starts[k], piece_end[k] + 1)
        ax.plot(np.arange(sl.stop - sl.start) * DT_FRAME, r[sl], color=col, lw=0.8, alpha=0.7)
    ax.set(title=f'pieces ending in {kind} ({len(pick)})', xlabel='time in piece (ps)')
axes[0].set_ylabel('$q$')
fig.tight_layout(); fig.savefig(f'{OUT}/q_along_pieces.png', dpi=110); plt.show()


## 7. q and the Other CVs

**Mean q per bin of each CV** (path-weighted, bins of equal weight, frames outside A and B) and **η²**, the share of the variance of q that this CV alone explains. A high η² means the CV is close to the committor by itself; a flat curve means it does not matter alone (it can still matter together with λ). The second figure ranks the CVs by η².

In [ ]:
qm = np.sum(W[free] * r[free]) / np.sum(W[free])
qvar = np.sum(W[free] * (r[free] - qm) ** 2) / np.sum(W[free])
eta = {}
ncol = 4
nrow = int(np.ceil(len(CV_NAMES) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3 * nrow), squeeze=False, sharey=True)
for ax, n in zip(axes.ravel(), CV_NAMES):
    x = d.X_all[free, CV_NAMES.index(n)]
    e = np.unique(np.quantile(x, np.linspace(0, 1, 41)))
    if len(e) < 3:
        ax.set_title(f'{n}: constant'); continue
    bi = np.clip(np.digitize(x, e) - 1, 0, len(e) - 2)
    wb = np.bincount(bi, W[free], len(e) - 1)
    qb = np.bincount(bi, W[free] * r[free], len(e) - 1) / np.maximum(wb, 1e-300)
    eta[n] = np.sum(wb * (qb - qm) ** 2) / np.sum(wb) / qvar
    ax.plot((e[1:] + e[:-1]) / 2, qb, '.-')
    ax.set(xlabel=label(n), title=f'{n}: $\\eta^2$ = {eta[n]:.2f}' + ('' if n in cv_names else ' (not fitted)'))
    ax.grid(alpha=0.3)
for ax in axes.ravel()[len(CV_NAMES):]:
    ax.axis('off')
for ax in axes[:, 0]:
    ax.set_ylabel('mean $q$')
fig.tight_layout(); fig.savefig(f'{OUT}/q_vs_cvs.png', dpi=110); plt.show()

eta_s = pd.Series(eta).sort_values()
fig, ax = plt.subplots(figsize=(6, 0.35 * len(eta_s) + 1))
ax.barh(eta_s.index, eta_s.values, color=['tab:blue' if n in cv_names else '0.6' for n in eta_s.index])
ax.set(xlabel=r'$\eta^2$ (share of Var q explained by the CV alone)', xlim=(0, 1))
fig.tight_layout(); fig.savefig(f'{OUT}/eta2.png', dpi=110); plt.show()


**Mean q in the (λ, CV) plane** for every CV, with the contours q = 0.1, 0.5, 0.9. Where the contours are vertical, q depends on λ alone; where they bend, the CV changes the fate at fixed distance (for example the angle or a pathway distance deciding whether a ligand at the pocket rim leaves or falls back).

In [ ]:
def binned_mean_2d(x, y, val, bins=(50, 40), min_w=1e-9, min_n=10):
    e_x = np.linspace(lambda_a, lambda_b, bins[0] + 1)
    e_y = y_edges(y, bins[1])
    n = np.histogram2d(x, y, (e_x, e_y))[0]
    sw = np.histogram2d(x, y, (e_x, e_y), weights=W[free])[0]
    sv = np.histogram2d(x, y, (e_x, e_y), weights=W[free] * val)[0]
    return e_x, e_y, np.where((n >= min_n) & (sw > min_w), sv / np.maximum(sw, 1e-300), np.nan)


others = [n for n in CV_NAMES if n != 'lambda']
ncol = 4
nrow = int(np.ceil(len(others) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.3 * ncol, 3.3 * nrow), squeeze=False, layout='constrained')
for ax, n in zip(axes.ravel(), others):
    ex, ey, M = binned_mean_2d(lam[free], d.X_all[free, CV_NAMES.index(n)], r[free])
    im = ax.pcolormesh(ex, ey, M.T, cmap='coolwarm', vmin=0, vmax=1)
    ax.contour((ex[1:] + ex[:-1]) / 2, (ey[1:] + ey[:-1]) / 2, M.T, levels=[0.1, 0.5, 0.9], colors='k', linewidths=0.8)
    ax.set(xlabel=label('lambda'), ylabel=label(n))
for ax in axes.ravel()[len(others):]:
    ax.axis('off')
fig.colorbar(im, ax=axes.ravel().tolist(), label='mean $q$', shrink=0.6)
fig.savefig(f'{OUT}/q_lambda_cv_maps.png', dpi=110); plt.show()


**Ligand position in the pocket frame.** Mean q in the (x, y) plane of the frame fixed to residues 189, 191 and 221, and the path-weighted density there (log). It shows through which side of the pocket the ligand leaves and where along that route q passes 0.5.

In [ ]:
if {'rel_x', 'rel_y'} <= set(CV_NAMES):
    xx, yy = d.X_all[free, CV_NAMES.index('rel_x')], d.X_all[free, CV_NAMES.index('rel_y')]
    ex = np.linspace(*np.quantile(xx, [0.005, 0.995]), 51)
    ey = np.linspace(*np.quantile(yy, [0.005, 0.995]), 51)
    n_ = np.histogram2d(xx, yy, (ex, ey))[0]
    sw = np.histogram2d(xx, yy, (ex, ey), weights=W[free])[0]
    sv = np.histogram2d(xx, yy, (ex, ey), weights=W[free] * r[free])[0]
    M = np.where(n_ >= 10, sv / np.maximum(sw, 1e-300), np.nan)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
    im = a1.pcolormesh(ex, ey, M.T, cmap='coolwarm', vmin=0, vmax=1)
    a1.contour((ex[1:] + ex[:-1]) / 2, (ey[1:] + ey[:-1]) / 2, M.T, levels=[0.1, 0.5, 0.9], colors='k', linewidths=0.8)
    fig.colorbar(im, ax=a1, label='mean $q$')
    im2 = a2.pcolormesh(ex, ey, np.log10(np.where(sw > 0, sw / sw.sum(), np.nan)).T, cmap='viridis')
    fig.colorbar(im2, ax=a2, label='log$_{10}$ weighted density')
    for a in (a1, a2):
        a.set(xlabel=label('rel_x'), ylabel=label('rel_y'), aspect='equal')
    fig.tight_layout(); fig.savefig(f'{OUT}/q_pocket_xy.png', dpi=110); plt.show()


**Conditional distribution of a CV at fixed λ.** −ln P(CV | λ) per λ column (each column shifted to 0 at its minimum), for the CVs in `COND_CVS`. It shows the channels at each distance (e.g. bound vs. turned ligand orientations, H-bonded or not), which the committor maps above then label with their fate. These are path-weighted densities of the pieces, not equilibrium free energies.

In [ ]:
COND_CVS = ['angle', 'n_hbonds', 'dA_216', 'dB_57']
COND_CVS = [n for n in COND_CVS if n in CV_NAMES]
fig, axes = plt.subplots(1, len(COND_CVS), figsize=(4.5 * len(COND_CVS), 3.6), squeeze=False)
for ax, n in zip(axes[0], COND_CVS):
    y = d.X_all[free, CV_NAMES.index(n)]
    H, xe, ye = np.histogram2d(lam[free], y, bins=(np.linspace(lambda_a, lambda_b, 51), y_edges(y, 40)), weights=W[free])
    P = H / np.maximum(H.sum(axis=1, keepdims=True), 1e-300)
    F = np.where(P > 0, -np.log(np.where(P > 0, P, 1)), np.nan)
    F -= np.nanmin(F, axis=1, keepdims=True)
    im = ax.pcolormesh(xe, ye, F.T, cmap='viridis_r', vmax=6)
    fig.colorbar(im, ax=ax, label=r'$-\ln P(\cdot \mid \lambda)$')
    ax.set(xlabel=label('lambda'), ylabel=label(n))
fig.tight_layout(); fig.savefig(f'{OUT}/cond_fe.png', dpi=110); plt.show()


## 8. Z_eq and F(q)

From the one-step increments of q along the pieces (path-weighted; steps across pieces left out): drift(q) and D(q) per q bin, Z_eq(q) = ∫ drift/D dq and F(q) = −Z_eq + ln D (the zero-flux condition (ln ρ)′ = drift/D − D′/D). Bins with fewer than `MIN_COUNT` weighted steps are left out.

- For the true committor the drift is zero (q is a martingale), so Z_eq ≈ 0 and F(q) ≈ ln D(q) + const. A Z_eq that is not flat means the fit is off.
- **These are not equilibrium free energies.** The weighted pieces are not an equilibrium sample (they start at the M crossings and carry a reactive current from A to B); compare runs and fits with them, not with experiment.
- The same is done in the natural coordinate q_n (`plots.transform_q2qn`, D = 1), and along λ itself.
- Error bands: bootstrap over **paths** (Poisson(1) weights per `group_traj`, so the pieces of one path go together).


In [ ]:
MIN_COUNT = 20
NBINS_ZEQ = 60


def comp_Zeq(r_traj, i_traj, w_traj, nbins=NBINS_ZEQ, min_count=MIN_COUNT, edges=None):
    rr = cp.asarray(r_traj, dtype=float)
    if edges is None:
        edges = cp.linspace(rr.min(), rr.max(), nbins + 1)
    nb = len(edges) - 1
    dr = rr[1:] - rr[:-1]
    ii = cp.asarray(i_traj)
    ww = cp.asarray(w_traj, dtype=float)[:-1] * (ii[1:] == ii[:-1])
    b = cp.clip(cp.searchsorted(edges, rr, side='right') - 1, 0, nb - 1)[:-1]
    n = cp.bincount(b, weights=ww, minlength=nb)
    s1 = cp.bincount(b, weights=ww * dr, minlength=nb)
    s2 = cp.bincount(b, weights=ww * dr ** 2, minlength=nb)
    m = n >= min_count
    ns = cp.where(m, n, 1.0)
    drift = s1 / ns
    D = cp.where(m, (s2 / ns - drift ** 2) / 2, cp.nan)
    zeq = cp.where(m, cp.cumsum(cp.where(m, drift / D, 0.0)) * (edges[1] - edges[0]), cp.nan)
    F = -(zeq - cp.log(D))
    F = F - cp.nanmin(F)
    mid = 0.5 * (edges[1:] + edges[:-1])
    return cp.asnumpy(mid), cp.asnumpy(zeq), cp.asnumpy(D), cp.asnumpy(F), edges


def bootstrap_Zeq(r_traj, n_boot=100, seed=0):
    # Poisson(1) weight per path (group_traj), the same for all its pieces
    _, inv = np.unique(d.group_traj, return_inverse=True)
    rs = np.random.default_rng(seed)
    mid, z0, D0, F0, edges = comp_Zeq(r_traj, d.i_traj, W)
    Z, Fb = [], []
    for _ in range(n_boot):
        wb = W * rs.poisson(1.0, inv.max() + 1)[inv]
        _, z, _, f, _ = comp_Zeq(r_traj, d.i_traj, wb, edges=edges)
        Z.append(z); Fb.append(f)
    return mid, z0, D0, F0, np.array(Z), np.array(Fb)


def band(ax, x, y0, boots, color, lab):
    ax.plot(x, y0, color=color, label=lab)
    ax.fill_between(x, np.nanpercentile(boots, 2.5, axis=0), np.nanpercentile(boots, 97.5, axis=0), color=color, alpha=0.2)


mid, z0, D0, F0, Zb, Fb = bootstrap_Zeq(r)
qn = optimalrcs.plots.transform_q2qn(q.r_traj, i_traj=q.i_traj, t_traj=q.t_traj, w_traj=cp.asarray(W), dt_sim=DT_FRAME)
mid_n, zn0, Dn0, Fn0, Znb, Fnb = bootstrap_Zeq(qn)
mid_l2, zl0, Dl0, Fl0, Zlb, Flb = bootstrap_Zeq(lam)

fig, axes = plt.subplots(2, 3, figsize=(16, 7.5))
band(axes[0, 0], mid, z0, Zb, 'tab:blue', '$Z_{eq}(q)$'); axes[0, 0].axhline(0, color='k', lw=0.6)
axes[0, 0].set(xlabel='$q$', ylabel='$Z_{eq}$', title='$Z_{eq}(q)$ (≈ 0 for the true committor)')
band(axes[0, 1], mid, F0, Fb, 'tab:red', '$F(q) = -Z_{eq} + \\ln D$')
axes[0, 1].set(xlabel='$q$', ylabel='$F$ ($k_BT$)', title='$F(q)$')
axes[0, 2].semilogy(mid, D0, 'k.-'); axes[0, 2].set(xlabel='$q$', ylabel='$D(q)$ per frame', title='diffusion of $q$')
band(axes[1, 0], mid_n, zn0, Znb, 'tab:blue', '$Z_{eq}(q_n)$'); axes[1, 0].axhline(0, color='k', lw=0.6)
axes[1, 0].set(xlabel='$q_n$ (natural, $D=1$)', ylabel='$Z_{eq}$', title='natural coordinate')
band(axes[1, 1], mid_n, Fn0, Fnb, 'tab:red', '$F(q_n)$')
axes[1, 1].set(xlabel='$q_n$', ylabel='$F$ ($k_BT$)', title='$F(q_n)$')
band(axes[1, 2], mid_l2, Fl0, Flb, 'tab:green', '$F(\\lambda)$')
axes[1, 2].set(xlabel=label('lambda'), ylabel='$F$ ($k_BT$)', title='$F(\\lambda)$ from the λ increments')
for a in axes.ravel():
    a.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(f'{OUT}/zeq_feq.png', dpi=110); plt.show()


## 9. Save

q per frame with its frames and all CVs, in the format of `load_tis_data_cupy.py`'s `committor.npz`, plus the CV names and the Z_eq profiles.

In [ ]:
np.savez(f'{OUT}/committor.npz', r=r, lam=lam, X_all=d.X_all, cv_names=np.array(CV_NAMES), fit_cvs=np.array(cv_names),
         i_traj=d.i_traj, t_traj=d.t_traj, group_traj=d.group_traj, path_weights=W,
         boundary0=d.boundary0, boundary1=d.boundary1, tis_dir=TIS_DIR, zqs_z=np.array(zqs),
         zeq_q=mid, zeq=z0, D_q=D0, F_q=F0, eta2_names=np.array(list(eta)), eta2=np.array(list(eta.values())))
print('saved', f'{OUT}/committor.npz')
